In [1]:
import fitz  # PyMuPDF
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from typing import List, Tuple
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

In [2]:
class InsuranceDocument:
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.text = self.load_pdf_with_mupdf()
        self.parsed_pages = self._split_parsed_text_into_pages()  # New helper method
        self.index, self.chunks, self.model = self.create_faiss_index()

    def load_pdf_with_mupdf(self) -> str:
        """Enhanced PDF loader with table handling and returns raw parsed text"""
        text = ""
        doc = fitz.open(self.file_path)
        
        for page_num, page in enumerate(doc, 1):
            text += f"\n=== Page {page_num} ===\n"
            
            if page_num >= 41:
                blocks = page.get_text("blocks")
                table_text = ""
                for block in blocks:
                    block_text = block[4]
                    table_text += f"{block_text}\n"
                text += f"TABLE CONTENT:\n{table_text}\n"
            else:
                page_text = page.get_text()
                text += f"{page_text}\n"
                
            if "Annexure" in page_text:
                text += "=== ANNEXURE SECTION START ===\n"
            elif "Exclusions" in page_text:
                text += "=== EXCLUSIONS SECTION START ===\n"
        
        doc.close()
        return text

    def _split_parsed_text_into_pages(self) -> dict:
        """Helper method to split parsed text into page dictionary"""
        pages = {}
        current_page = None
        current_content = []
        
        for line in self.text.split('\n'):
            if line.startswith("=== Page"):
                if current_page is not None:
                    pages[current_page] = '\n'.join(current_content)
                current_page = int(line.split()[2])
                current_content = []
            else:
                current_content.append(line)
        
        if current_page is not None:
            pages[current_page] = '\n'.join(current_content)
            
        return pages

    def create_chunks(self, text: str, chunk_size: int = 2000, overlap: int = 400) -> List[str]:
        """Enhanced chunking with context preservation"""
        chunks = []
        lines = text.split('\n')
        current_chunk = []
        current_length = 0
        
        for line in lines:
            if line.startswith("==="):
                if current_chunk:
                    chunks.append('\n'.join(current_chunk))
                current_chunk = [line]
                current_length = len(line)
                continue
                
            if "TABLE CONTENT:" in line:
                chunk_size = 1000
            else:
                chunk_size = 2000
                
            if current_length + len(line) > chunk_size:
                chunks.append('\n'.join(current_chunk))
                overlap_start = max(0, len(current_chunk) - overlap)
                current_chunk = current_chunk[overlap_start:] + [line]
                current_length = sum(len(l) for l in current_chunk)
            else:
                current_chunk.append(line)
                current_length += len(line)
                
        if current_chunk:
            chunks.append('\n'.join(current_chunk))
        
        return chunks

    def create_faiss_index(self) -> Tuple[faiss.IndexFlatL2, List[str], SentenceTransformer]:
        """Create FAISS index with embeddings"""
        model = SentenceTransformer('all-MiniLM-L6-v2')
        chunks = self.create_chunks(self.text)
        chunk_embeddings = model.encode(chunks)
        dimension = chunk_embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(np.array(chunk_embeddings))
        return index, chunks, model


In [3]:
# File Upload Widget
uploader = widgets.FileUpload(
    description='Upload PDF',
    accept='.pdf',
    multiple=False
)
display(uploader)

FileUpload(value=(), accept='.pdf', description='Upload PDF')

In [4]:
def process_document(btn):
    clear_output()
    display(uploader)
    
    if not uploader.value:
        print("Please upload a PDF file first")
        return
    
    # Save uploaded file
    uploaded_file = next(iter(uploader.value))
    with open("temp.pdf", "wb") as f:
        f.write(uploaded_file['content'])
    
    # Process document
    print("Processing document...")
    insurance_doc = InsuranceDocument("temp.pdf")
    
    # Show parsing results
    print("\n=== PARSING RESULTS ===")
    print(f"Total pages parsed: {len(insurance_doc.parsed_pages)}")
    
    # Show sample pages
    print("\n=== SAMPLE PAGE PARSING ===")
    for page_num in [1, 40, 41, 42]:
        if page_num in insurance_doc.parsed_pages:
            print(f"\nPage {page_num} Content:")
            print(insurance_doc.parsed_pages[page_num][:500] + "...")  # Show first 500 chars
    
    # Show chunking results
    print("\n=== CHUNKING RESULTS ===")
    print(f"Total chunks created: {len(insurance_doc.chunks)}")
    print("\nSample Chunks:")
    for i, chunk in enumerate(insurance_doc.chunks[:3]):
        print(f"\nChunk {i+1} ({len(chunk)} characters):")
        print(chunk[:300] + "...")  # Show first 300 characters
    
    # Store document in variable for further inspection
    global processed_doc
    processed_doc = insurance_doc
    print("\nDocument processing complete! Use 'processed_doc' variable to inspect:")



In [5]:
process_btn = widgets.Button(description="Process Document")
process_btn.on_click(process_document)
display(process_btn)

Button(description='Process Document', style=ButtonStyle())

In [6]:
# Example manual inspection cells (run after processing document)
if 'processed_doc' in globals():
    # View specific page
    print("Page 41 Full Content:")
    display(processed_doc.parsed_pages.get(41, "Page not found"))
    
    # View chunk distribution
    chunk_lengths = [len(chunk) for chunk in processed_doc.chunks]
    print(f"\nChunk length distribution (avg: {np.mean(chunk_lengths):.0f} chars, min: {min(chunk_lengths)}, max: {max(chunk_lengths)})")
    
    # View table handling
    table_chunks = [chunk for chunk in processed_doc.chunks if "TABLE CONTENT:" in chunk]
    print(f"\nFound {len(table_chunks)} table chunks")
    if table_chunks:
        print("\nFirst Table Chunk:")
        print(table_chunks[0][:500] + "...")

In [9]:
# Specify the local path to your PDF document
local_pdf_path = "/Users/a1/Documents/GitHub/pdf_processing/NBHTGBP22011V012223.pdf"  # Change this to your actual file path

# Process the document
insurance_doc = InsuranceDocument(local_pdf_path)

# Show parsing results
print("\n=== PARSING RESULTS ===")
print(f"Total pages parsed: {len(insurance_doc.parsed_pages)}")

# Show sample pages
print("\n=== SAMPLE PAGE PARSING ===")
for page_num in [50,51,52,53,54,55,56,57,58]:
    if page_num in insurance_doc.parsed_pages:
        print(f"\nPage {page_num} Content:")
        print(insurance_doc.parsed_pages[page_num][:])  # Show first 500 chars


=== PARSING RESULTS ===
Total pages parsed: 58

=== SAMPLE PAGE PARSING ===

Page 50 Content:
TABLE CONTENT:
 

Product Name: Travel infinity | Product UIN: NBHTGBP22011V012223 

 
Copy of the application for bail and the evidence of cost incurred towards 
procurement of such bail 
Financial Emergency Cash 
Mugging Benefit 

 
Copy of complaint lodged with police authorities or FIR   

Loan protector 
Travel loan secure 

 
Medical reports giving the details of the Accident, nature of the injury, 
the extent of disability (if applicable) and the details of the treatment 
provided  
 
Death certificate (if applicable) 
 
Postmortem report, if conducted 
 
Police report 
 
Loan documents from bank/ financial institution 
Sports Equipment hire 
Sports Equipment Cover 

 
Property irregularity report issued by the appropriate authority stating 
the scheduled time of delivery and actual time of delivery of the 
Checked-in-Baggage. 
 
Voucher of the Common Carrier for the compensat